---
title: "Queue from Two Stacks"
language: "Python"
difficulty: "Beginner"
topic: "Data structures"
slug: "queue-from-two-stacks"
---

## Problem Statement

Implement a class `TwoStackQueue` that behaves like a **first-in, first-out (FIFO) queue** using only two Python lists as internal storage. You may **not** use `collections.deque` or any other queue primitive.

**Methods to implement:**

| Method | Description |
|---|---|
| `enqueue(value)` | Add `value` to the back of the queue. |
| `dequeue() -> any` | Remove and return the value at the front of the queue. Raises `IndexError` with the message `"dequeue from empty queue"` if the queue is empty. |
| `peek() -> any` | Return (but do not remove) the front value. Raises `IndexError` with the message `"peek from empty queue"` if empty. |
| `is_empty() -> bool` | Return `True` if the queue has no elements. |
| `size() -> int` | Return the number of elements currently in the queue. |

**Constraint:** The only list operations you may use are `append` (push to top of stack) and `pop()` with no arguments (pop from top of stack).

```python
class TwoStackQueue:
    def __init__(self):
        self._inbox = []   # newly enqueued items go here
        self._outbox = []  # items ready to be dequeued come from here

    def enqueue(self, value) -> None:
        # TODO
        pass

    def dequeue(self):
        # TODO
        pass

    def peek(self):
        # TODO
        pass

    def is_empty(self) -> bool:
        # TODO
        pass

    def size(self) -> int:
        # TODO
        pass
```

## Expected Output / Test Cases

```python
q = TwoStackQueue()

# is_empty and size on a fresh queue
assert q.is_empty() is True
assert q.size() == 0

# Basic enqueue / dequeue (FIFO order)
q.enqueue(1)
q.enqueue(2)
q.enqueue(3)
assert q.size() == 3
assert q.dequeue() == 1   # first in, first out
assert q.dequeue() == 2
assert q.size() == 1

# peek does not remove the element
assert q.peek() == 3
assert q.size() == 1

# Interleaved enqueue and dequeue
q.enqueue(4)
q.enqueue(5)
assert q.dequeue() == 3
assert q.dequeue() == 4
assert q.dequeue() == 5
assert q.is_empty() is True

# Error cases
import traceback
try:
    q.dequeue()
    assert False, "Expected IndexError"
except IndexError as e:
    assert str(e) == "dequeue from empty queue"

try:
    q.peek()
    assert False, "Expected IndexError"
except IndexError as e:
    assert str(e) == "peek from empty queue"
```

## Real-World Context

The two-stack queue is a classic data-structures interview problem, but it has practical significance: it is the pattern underlying many producer-consumer pipelines where you want to *batch* incoming work (inbox stack) and *process* it in order (outbox stack) without locking the whole structure on every operation. Amortised analysis shows that each element is moved at most once between stacks, giving O(1) amortised cost per operation — a useful intuition for evaluating lazy-evaluation and buffering strategies in real systems.

## Hints

<details>
<summary>Hint 1 — Conceptual nudge</summary>

A stack is LIFO (last in, first out); a queue is FIFO (first in, first out). If you push items onto one stack and then pop them all onto a second stack, the order reverses — and a reversed-reversed order is the original order. Think about when it makes sense to transfer items from one stack to the other.

</details>

<details>
<summary>Hint 2 — More direct</summary>

Use `_inbox` for enqueue: always `append` to it. Use `_outbox` for dequeue/peek: when `_outbox` is empty, pour all items from `_inbox` into `_outbox` by repeatedly popping from `_inbox` and appending to `_outbox`. Now `_outbox.pop()` removes the oldest item. Only transfer when `_outbox` is empty — this is what keeps the amortised cost low.

</details>

<details>
<summary>Hint 3 — Near-solution guidance</summary>

```python
def _transfer(self):
    if not self._outbox:                  # only move when outbox is empty
        while self._inbox:
            self._outbox.append(self._inbox.pop())

def dequeue(self):
    self._transfer()
    if not self._outbox:
        raise IndexError("dequeue from empty queue")
    return self._outbox.pop()
```

</details>

## Reference Solution

```python
class TwoStackQueue:
    def __init__(self):
        self._inbox = []   # receives newly enqueued items (top = most recent)
        self._outbox = []  # serves dequeue/peek requests (top = oldest item)

    def _transfer(self) -> None:
        """Move all items from inbox to outbox, reversing their order.
        Called lazily — only when outbox is empty and we need to serve a request."""
        if not self._outbox:
            while self._inbox:
                # Popping from the top of inbox and pushing to outbox reverses order,
                # so the oldest item ends up at the top of outbox.
                self._outbox.append(self._inbox.pop())

    def enqueue(self, value) -> None:
        self._inbox.append(value)

    def dequeue(self):
        self._transfer()
        if not self._outbox:
            raise IndexError("dequeue from empty queue")
        return self._outbox.pop()

    def peek(self):
        self._transfer()
        if not self._outbox:
            raise IndexError("peek from empty queue")
        return self._outbox[-1]   # look at top without removing

    def is_empty(self) -> bool:
        return not self._inbox and not self._outbox

    def size(self) -> int:
        return len(self._inbox) + len(self._outbox)
```

## Follow-Up Challenge

Add a `__repr__` method that returns a human-readable string showing the queue contents from front to back, e.g. `TwoStackQueue([1, 2, 3])` where `1` is next to be dequeued. Do this **without modifying** `_inbox` or `_outbox` — compute the view from their current state only.